#  `lesson07`:  Monitoring Online Data Sources

-   Acquire data from a static or streaming source using a data-scraping tool.
-   Work with well-formed or arbitrarily formatted input data.
-   Visualize continuously-updated data.

### Internet Data



Most data on the Internet are available in plain-text or binary form.  

You may have to worry about encodings, or the particular form the data are stored in, but this is becoming a less common concern as time goes on.

Sometimes you need to access a _streaming service_ such as a socket or API.  The defining characteristic of these is that the data are posted in real time (continuously or periodically) and so your program needs to monitor the data source for new data.

There are two ways to acquire data from the web as plain-text:  scrape the data yourself using [`urllib`](https://docs.python.org/3/library/urllib.html) or [`requests`](http://docs.python-requests.org/en/master/), or use a scraping tool like [Beautiful Soup](https://www.crummy.com/software/BeautifulSoup/).

Scraping the data directly can work well if the data are consistently formatted in a particular tabular way.  For instance, the National Oceanic and Atmospheric Administration (NOAA) publishes [historical weather data online](https://www1.ncdc.noaa.gov/pub/data/ghcn/daily/) in a regular format.  For the Champaign—Willard Airport station, the code is `USW00094870` and the data are in the form:

    USW00094870199706TMAX  156  M  217  M  183  M  222  M  233  M  217  M  206  M  178  M  244  M  261  M  
    USW00094870199706TMIN  111  M  139  M  128  M  106  M  106  M  150  M  139  M  133  M  122  M  111  M    
    USW00094870199707TMAX  306  M  328  M  244  M  206  M  244  M  272  M  289  M  306  M  217  M  250  M    
    USW00094870199707TMIN  167  M  167  M  150  M  111  M  106  M  128  M  156  M  178  M  144  M  139  M    

Each line begins with the station identifier, then the date in `YYYYMM` format, then the type of data begin reported (`TMAX`, high temperature, or `TMIN`, low temperature).  Note that temperatures are [reported](https://www1.ncdc.noaa.gov/pub/data/ghcn/daily/readme.txt) in _tenths_ of a degree centigrade.  Each line represents a month (the lines above are truncated).

For the first part of this project, you will load the data, parse the daily temperatures, and plot the data points over time.

In [ ]:
url = 'https://www1.ncdc.noaa.gov/pub/data/ghcn/daily/all/USW00094870.dly'
import requests
data_source = requests.get( url ).text

print( data_source )

In [ ]:
data_strings = data_source.split( '\n' )[ ::2 ]  # take every other line, so just high temperatures
data = [ data_string.split() for data_string in data_strings ]

In [ ]:
import datetime
import numpy as np

data_dates = []
data_temps = []
for line in data:
    for day,piece in enumerate( line[ 1::2 ] ):
        try:
            # Read off month and year, then add in date.
            year  = int( line[ 0 ][ 11:15 ] )
            month = int( line[ 0 ][ 15:17 ] )
            data_dates.append( datetime.date( year,month,day+1 ) )
        except Exception as exc:
            continue
        try:
            data_temps.append( float( piece ) / 10 )
        except Exception as exc:
            data_temps.append( np.nan )
            continue

In [ ]:
import pandas as pd
data_set = pd.DataFrame( data_temps,index=data_dates )
len( data_set )

In [ ]:
import matplotlib.pyplot as plt
%matplotlib inline
#plt.plot( data_set[0:],'r.' )
plt.plot( data_set[10000:10760],'r.' )
plt.ylim( -40,40 )
plt.show()

The data may need to be cleaned as well.  There may be strange `NULL` values lying out-of-bounds, such as `-9999`.  If that is not the case with this data set, you may skip to the Beautiful Soup section below.

In [ ]:
type( data_set )

In [ ]:
data_set[ data_set[ 0 ] != -999.9 ]

In [ ]:
plt.plot( data_set[ data_set[ 0 ] != -999.9 ][10000:10365],'r.' )
plt.show()

#### Data Scraping with Beautiful Soup

Data scraping is a fairly manual process the first time around, although once the salient features have been identified it can be automated to some extent.

We will use [Beautiful Soup](https://www.crummy.com/software/BeautifulSoup/) to extract the data from the raw HTML string that `requests` gives us.  BS gives us the ability to search for features and iterate over them, a more comfortable data structure than a raw string.

In [ ]:
!pip install beautifulsoup4

In [ ]:
import requests
data_url  = 'https://w1.weather.gov/data/obhistory/KCMI.html'
data_html = requests.get( data_url )

In [ ]:
from bs4 import BeautifulSoup
soup = BeautifulSoup( data_html.text,'html.parser' )

print( soup.prettify() )

BS calls the resulting data structure "soup," because it's a pile of all of the data in a heap.

We can search through to find features by HTML attributes or tags.  We know from examination that the data we are interested in are contained in table rows, represented in HTML by the tag `<tr>`, and that they have one of two background colors, `#f5f5f5` or `#eeeeee`.  These are both somewhat arbitrary elements to notice, but such is the nature of data scraping.

In [ ]:
# Find all table rows in the page.
for tr in soup.find_all( 'tr' ):
    # Look for table rows with a specified background color.
    if 'bgcolor' in tr.attrs and ( '#eeeeee' in tr[ 'bgcolor' ] or '#f5f5f5' in tr[ 'bgcolor' ] ):
        # Loop over
        print( tr )

We will collect these values into data arrays for each according to the header row.  Table header rows are identified with `<th>`, and in this case repeat at the beginning and the end of the page.  So first, let's get the header names so we know where to put the `<tr>` data.

In [ ]:
entries = []

# Find all table headers in the page.
for th in soup.find_all( 'th' ):
    entries.append( th.text )

# Every entry appears twice so cut the list in half.
entries = entries[ 0:len( entries )//2 ]
print( entries )

Each entry will correspond, in the end, to an array of values.  Right now, the titles are a bit messy due to how the table header is actually structured, with odd nested header lines.  This means that some entries are superfluous, so we can remove them.  We also need to reorder things to be consistent with the table column order.

<table>
<tr bgcolor="#b0c4de" align="center"><th rowspan="3" width="17">D<br>a<br>t<br>e</th><th rowspan="3" width="32">Time<br>(cst)</th>
							<th rowspan="3" width="80">Wind<br>(mph)</th><th rowspan="3" width="40">Vis.<br>(mi.)</th><th rowspan="3" width="80">Weather</th><th rowspan="3" width="65">Sky Cond.</th>
							<th colspan="4">Temperature (ºF)</th><th rowspan="3" width="65">Relative<br>Humidity</th><th rowspan="3" width="80">Wind<br>Chill<br>(°F)</th><th rowspan="3" width="80">Heat<br>Index<br>(°F)</th><th colspan="2">Pressure</th><th colspan="3">Precipitation (in.)</th></tr>
<tr bgcolor="#b0c4de" align="center"><th rowspan="2" width="45">Air</th><th rowspan="2" width="26">Dwpt</th><th colspan="2">6 hour</th>
							<th rowspan="2" width="40">altimeter<br>(in)</th><th rowspan="2" width="40">sea level<br>(mb)</th><th rowspan="2" width="24">1 hr</th>
							<th rowspan="2" width="24">3 hr</th><th rowspan="2" width="30">6 hr</th></tr>
<tr bgcolor="#b0c4de" align="center"><th width="26">Max.</th><th width="26">Min.</th></tr>
</table>

In [ ]:
del entries[ entries.index( 'Temperature (ºF)' ) ]
del entries[ entries.index( 'Pressure' ) ]
del entries[ entries.index( 'Precipitation (in.)' ) ]
del entries[ entries.index( '6 hour' ) ]

del entries[ entries.index( 'Air' ) ]
entries.insert( entries.index( 'Sky Cond.' )+1,'Air' )
del entries[ entries.index( 'Dwpt' ) ]
entries.insert( entries.index( 'Sky Cond.' )+2,'Dwpt' )
del entries[ entries.index( 'Max.' ) ]
entries.insert( entries.index( 'Sky Cond.' )+3,'Max.' )
del entries[ entries.index( 'Min.' ) ]
entries.insert( entries.index( 'Sky Cond.' )+4,'Min.' )

In [ ]:
data = { key:[] for key in entries }

# Find all table rows in the page.
for tr in soup.find_all( 'tr' ):
    # Look for table rows with a specified background color.
    if 'bgcolor' in tr.attrs and ( '#eeeeee' in tr[ 'bgcolor' ] or '#f5f5f5' in tr[ 'bgcolor' ] ):
        # Loop over each entry in order and append it to the appropriate list.
        for index,td in enumerate( tr.find_all( 'td' ) ):
            data[ entries[ index ] ].append( td.text )

The data are now all strings instead of numbers, but we have successfully extracted the table using Beautiful Soup.  The next step is to convert the dictionary `data` into a Pandas DataFrame.

In [ ]:
import pandas as pd

data_df = pd.DataFrame.from_dict( data )

data_df

At a minimum, we need to do the following to complete our data cleaning:

1.  Convert each numeric column to `float` or `int` (each one is still `str`).
    1.  Convert blanks and `'NA'` entries to NaNs.
2.  Convert the `'Date'` and `'Time(cst)'` columns to a Python `datetime` object, column `'Date/Time'`
3.  Renaming the headers for `'Temperature'`, `'Pressure'`, etc.
4.  Handle `'Wind(mph)'` entries as vectors instead of notations.

We will then display the data graphically.

#### 1.  Convert each numeric column to `float` or `int` (each one is still `str`).

In [ ]:
import numpy as np

for entry in [ 'Vis.(mi.)','RelativeHumidity','WindChill(°F)','HeatIndex(°F)','Air','Dwpt','altimeter(in)','sea level(mb)','1 hr','3 hr','6 hr','Max.','Min.' ]:
    data_df[ entry ] = data_df[ entry ].replace( '',np.nan )
    data_df[ entry ] = data_df[ entry ].replace( 'NA',np.nan )
    try:
        data_df[ entry ] = data_df[ entry ].astype( float )
    except ValueError:
        data_df[ entry ] = data_df[ entry ].str.rstrip( '%' ).astype( 'float' ) / 100.0
data_df = data_df.fillna(0)

In [ ]:
data_df

#### 2.  Convert the `'Date'` and `'Time(cst)'` columns to a Python `datetime` object, column `'Date/Time'`

The tricky thing about converting the date is keeping track of the proper month.  If we just specify the day number, Pandas assumes we are counting from January 1900.  We have to specify this month as the origin.

In [ ]:
import datetime

date_offset = int( data_df[ 'Date' ][ 0 ] )
date_origin = datetime.date.today() - datetime.timedelta( days=date_offset )

data_df[ 'Date' ] = pd.to_numeric( data_df[ 'Date' ] )
data_df[ 'Date' ] = pd.to_datetime( data_df[ 'Date' ],unit='D',
                                    origin=pd.Timestamp( date_origin.strftime( '%Y-%m-%d' ) ) )

In [ ]:
data_df[ 'Date' ] = data_df[ 'Date' ] + pd.to_timedelta( data_df[ 'Time(cdt)' ].astype( str ) + ':00' )
data_df = data_df.drop( columns='Time(cdt)' )

#### 3.  Renaming the headers for `'Temperature'`, `'Pressure'`, etc.

In [ ]:
data_df = data_df.rename( columns={ 'Air':'Temperature(°F)',
                                    'Dwpt':'DewpointTemperature(°F)',
                                    '1 hr':'Precipitation1Hr(In)',
                                    '3 hr':'Precipitation3Hr(In)',
                                    '6 hr':'Precipitation6Hr(In)',
                                    'altimeter(in)':'PressureAltimeter(In)',
                                    'sea level(mb)':'PressureSeaLevel(mb)',
                                    'Max.':'Temperature6HrMax(°F)',
                                    'Min.':'Temperature6HrMin(°F)'
                                  } )

In [ ]:
import matplotlib.pyplot as plt
import matplotlib as mpl
mpl.rcParams[ 'figure.figsize' ] = (10,3)

data_df.plot( x='Date',style='.' )
plt.ylim( ( 0,50 ) )
plt.legend( loc='right',bbox_to_anchor=(1.65,0.5),ncol=2 )
plt.show()

In [ ]:
data_df

#### 4.  Handle `'Wind(mph)'` entries as vectors instead of notations.

Wind bearings are reported in 45° increments by direction, along with integer windspeeds.  Sometimes there is an additional term such as `'NE 15 G 23'`, where `'G 23'` refers to the gust speed, as opposed to the sustained windspeed.  We can strip these out for now, and interpret `'Calm'` as zero windspeed, any direction.

We will represent the wind bearing as a vector, which is a directional quantity (that is, it has direction and magnitude).  Probably the simplest way to do this is to use a NumPy array to hold the value.  We will predefine eight directional unit vectors for the directions, and then multiply each one times the windspeed to create the vector.

In [ ]:
import numpy as np

unit_vectors = { 'N'  : np.array( ( 0,1 ) ),
                 'NE' : np.array( ( +2**0.5,+2**0.5 ) ) / 2,
                 'E'  : np.array( ( 1,0 ) ),
                 'SE' : np.array( ( +2**0.5,-2**0.5 ) ) / 2,
                 'S'  : np.array( ( 0,-1 ) ),
                 'SW' : np.array( ( -2**0.5,-2**0.5 ) ) / 2,
                 'W'  : np.array( ( -1,0 ) ),
                 'NW' : np.array( ( -2**0.5,+2**0.5 ) ) / 2,
                 'Calm' : np.zeros( ( 2, ) )
}

mpl.rcParams[ 'figure.figsize' ] = ( 3,3 )
for vector in unit_vectors:
    plt.plot( unit_vectors[ vector ][ 0 ],unit_vectors[ vector ][ 1 ],'o' )
plt.show()

The next step is to write a function to parse an entry into one of these unit vectors times the windspeed magnitude.  We will pass these into the dataframe as separate columns `'WindX'` and `'WindY'` to ease plotting.

In [ ]:
def parseWindspeed( string ):
    unit_vectors = { 'N'  : np.array( ( 0,1 ) ),
                     'NE' : np.array( ( +2**0.5,+2**0.5 ) ) / 2,
                     'E'  : np.array( ( 1,0 ) ),
                     'SE' : np.array( ( +2**0.5,-2**0.5 ) ) / 2,
                     'S'  : np.array( ( 0,-1 ) ),
                     'SW' : np.array( ( -2**0.5,-2**0.5 ) ) / 2,
                     'W'  : np.array( ( -1,0 ) ),
                     'NW' : np.array( ( -2**0.5,+2**0.5 ) ) / 2,
                     'Calm' : np.zeros( ( 2, ) )
    }
    
    elements = string.split( ' ' )
    if elements[ 0 ] == 'Calm':
        return unit_vectors[ 'Calm' ]
    elif elements[ 0 ] not in unit_vectors.keys():
        return unit_vectors[ 'Calm' ]
    else:
        direction = elements[ 0 ]
        magnitude = int( elements[ 1 ] )
        return unit_vectors[ direction ] * magnitude

In [ ]:
data_df[ [ 'WindX','WindY' ] ] = pd.DataFrame( data_df[ 'Wind(mph)' ].apply( parseWindspeed ).values.tolist(),index=data_df.index )

In [ ]:
data_df.plot.scatter( x=[ 'WindX' ],y=[ 'WindY' ],marker='.' )

In [ ]:
data_df.columns

Let's summarize our data in a single static view.

In [ ]:
mpl.rcParams[ 'figure.figsize' ] = ( 15,15 )
fig,ax = plt.subplots( 2,2 )

# Temperature
ax[ 0,0 ].set_title( 'Temperatures' )
ax[ 0,0 ].plot( data_df[ 'Date' ],data_df[ 'Temperature(°F)' ],'r-' )
ax[ 0,0 ].plot( data_df[ 'Date' ],data_df[ 'HeatIndex(°F)' ],'r.' )
ax[ 0,0 ].plot( data_df[ 'Date' ],data_df[ 'WindChill(°F)' ],'b.' )
ax[ 0,0 ].plot( data_df[ 'Date' ],data_df[ 'DewpointTemperature(°F)' ],'g-' )
ax[ 0,0 ].set_ylim( ( -20,120 ) )
ax[ 0,0 ].legend()

# Pressure
ax[ 1,0 ].set_title( 'Pressure & Humidity' )
ax[ 1,0 ].plot( data_df[ 'Date' ],data_df[ 'PressureAltimeter(In)' ],'m.' )
ax1b = ax[ 1,0 ].twinx()
ax1b.plot( data_df[ 'Date' ],data_df[ 'RelativeHumidity' ],'c.' )
# Adding a shared legend across two axes is nontrivial:  see https://stackoverflow.com/questions/5484922/secondary-axis-with-twinx-how-to-add-to-legend

# Precipitation
ax[ 0,1 ].set_title( 'Precipitation' )
ax[ 0,1 ].plot( data_df[ 'Date' ],data_df[ 'Precipitation1Hr(In)' ],'k.' )
ax[ 0,1 ].plot( data_df[ 'Date' ],data_df[ 'Precipitation3Hr(In)' ],'y.' )
ax[ 0,1 ].plot( data_df[ 'Date' ],data_df[ 'Precipitation6Hr(In)' ],'c.' )
ax[ 0,1 ].legend()

# Windspeed and Direction
ax[ 1,1 ].set_title( 'Wind' )
ax[ 1,1 ].scatter( data_df[ 'WindX' ],data_df[ 'WindY' ] )
ax[ 1,1 ].set_xlim( ( -40,40 ) )
ax[ 1,1 ].set_ylim( ( -40,40 ) )

plt.show()

### Streaming Data Dashboard

A _streaming data source_ is a continuously-updated socket or endpoint from which data can be obtained.  Sometimes this is an API which responds to individual requests ([RESTful APIs](https://en.wikipedia.org/wiki/Representational_state_transfer)), sometimes this is a dynamically updated web page.

Such data sources can provide many data of consequence:

-   video, audio, or image sources
-   meteorological data (historical and forecast data)
-   financial data

Aside from video and audio data sources, most streaming data sources do not update terribly quickly (that is, multiple times per second).  While we could treat such a data source as a static page, it's more interesting to build a dashboard which monitors the streaming data source and updates in real time as new data roll in.  (Financial data also update very frequently, but these are generally offered by for-profit services.)

#### Building a Dashboard

A _dashboard_ is a representation of several related data components, potentially including the history, in a layout and style reminiscent of an automobile dashboard.

There are [a few options](https://blog.sicara.com/bokeh-dash-best-dashboard-framework-python-shiny-alternative-c5b576375f7f) for making Python dashboards:

-   [Dash by Plotly](https://plot.ly/python/dashboard/))
-   [Bokeh](https://bokeh.pydata.org/en/latest/docs/gallery.html#gallery)
-   [Bowtie](https://github.com/jwkvam/bowtie)

We'll use Bokeh for our dashboard.  Bokeh has the advantage of more interactions but can be more difficult to prepare data to work with.

Let's look at one of the canned examples, [`weather`](https://github.com/bokeh/bokeh/tree/master/examples/app/weather).  A Bokeh server app requires all of the files to be located in the same folder, with the initiating code stored in `main.py`.

In [ ]:
%%file main.py
from os.path import join, dirname
import datetime

import pandas as pd
from scipy.signal import savgol_filter

from bokeh.io import curdoc
from bokeh.layouts import row, column
from bokeh.models import ColumnDataSource, DataRange1d, Select
from bokeh.palettes import Blues4
from bokeh.plotting import figure

STATISTICS = ['record_min_temp', 'actual_min_temp', 'average_min_temp', 'average_max_temp', 'actual_max_temp', 'record_max_temp']

def get_dataset(src, name, distribution):
    df = src[src.airport == name].copy()
    del df['airport']
    df['date'] = pd.to_datetime(df.date)
    # timedelta here instead of pd.DateOffset to avoid pandas bug < 0.18 (Pandas issue #11925)
    df['left'] = df.date - datetime.timedelta(days=0.5)
    df['right'] = df.date + datetime.timedelta(days=0.5)
    df = df.set_index(['date'])
    df.sort_index(inplace=True)
    if distribution == 'Smoothed':
        window, order = 51, 3
        for key in STATISTICS:
            df[key] = savgol_filter(df[key], window, order)

    return ColumnDataSource(data=df)

def make_plot(source, title):
    plot = figure(x_axis_type="datetime", plot_width=800, tools="", toolbar_location=None)
    plot.title.text = title

    plot.quad(top='record_max_temp', bottom='record_min_temp', left='left', right='right',
              color=Blues4[2], source=source, legend="Record")
    plot.quad(top='average_max_temp', bottom='average_min_temp', left='left', right='right',
              color=Blues4[1], source=source, legend="Average")
    plot.quad(top='actual_max_temp', bottom='actual_min_temp', left='left', right='right',
              color=Blues4[0], alpha=0.5, line_color="black", source=source, legend="Actual")

    # fixed attributes
    plot.xaxis.axis_label = None
    plot.yaxis.axis_label = "Temperature (F)"
    plot.axis.axis_label_text_font_style = "bold"
    plot.x_range = DataRange1d(range_padding=0.0)
    plot.grid.grid_line_alpha = 0.3

    return plot

def update_plot(attrname, old, new):
    city = city_select.value
    plot.title.text = "Weather data for " + cities[city]['title']

    src = get_dataset(df, cities[city]['airport'], distribution_select.value)
    source.data.update(src.data)

city = 'Austin'
distribution = 'Discrete'

cities = {
    'Austin': {
        'airport': 'AUS',
        'title': 'Austin, TX',
    },
    'Boston': {
        'airport': 'BOS',
        'title': 'Boston, MA',
    },
    'Seattle': {
        'airport': 'SEA',
        'title': 'Seattle, WA',
    }
}

city_select = Select(value=city, title='City', options=sorted(cities.keys()))
distribution_select = Select(value=distribution, title='Distribution', options=['Discrete', 'Smoothed'])

df = pd.read_csv(join(dirname(__file__), 'data/2015_weather.csv'))
source = get_dataset(df, cities[city]['airport'], distribution)
plot = make_plot(source, "Weather data for " + cities[city]['title'])

city_select.on_change('value', update_plot)
distribution_select.on_change('value', update_plot)

controls = column(city_select, distribution_select)

curdoc().add_root(row(plot, controls))
curdoc().title = "Weather"

The basic idea is that the Python contents (in `main.py` and elsewhere) structures the app, which runs on the Bokeh server (here running locally rather than across the Internet).

The app is run via Bokeh by invoking the folder containing `main.py` with `bokeh serve`.  A browser window should open up with the interactive widget displaying the data.

In [ ]:
!bokeh serve --show .

While that cell is running `[*]`, you won't be able to run any more Python code in this notebook.  Select `Kernel`→`Interrupt` to stop it running so that you can continue.

While most Bokeh apps are intended to stand on their own, Bokeh also integrates nicely with the Jupyter notebook.  We will use this latter capability in the current lesson so you can view the updates immediately in this window.

#### A Dashboard for Weather Data

In this part of the lesson, you will build a dashboard tool to request and monitor various weather data from one or more local weather stations.

The data a weather station provides include:

-   temperature
-   dew point
-   wind speed (but not wind direction)
-   atmospheric pressure
-   precipitation
-   solar radiation, uv index

You can see a dashboard view of these data [online at Weather Underground](https://www.wunderground.com/personal-weather-station/dashboard?ID=KILURBAN12#history).  We'll build a similar tool, but use some different widgets to represent the data.

-   Create a folder `urbana`.

In [ ]:
!mkdir -p urbana/

First, we are going to scrape some data down from the personal weather station `KILURBAN12`.

We will acquire continuously updated data from Weather Underground.  Formerly, WU offered an API for more direct access, but that functionality has been retired.  Instead, we will scrape the data manually, store the data in a local buffer, and update periodically.  The weather station we will use, `KILURBAN12` provided by `UrbanaTony`, updates every five minutes.

In [ ]:
import requests

data_url  = 'https://www.wunderground.com/personal-weather-station/dashboard?ID=KILURBAN12'
data_html = requests.get( data_url )

from bs4 import BeautifulSoup
soup = BeautifulSoup( data_html.text,'html.parser' )

print( soup.prettify() )

If you scroll through the `soup`, you can see that there are a _lot_ of data represented, most in a verbose format with multiple metadata entries per data point.  We are primarily interested in a block in a `<script>` near the end called `'app-root-state'`, which contains the data in the plots.  We will scrape those data for a particular date, reconstruct them into an array, and plot them using Bokeh as the dashboard.

1.  First, locate the appropriate script block:

In [ ]:
soup.find_all('script',id='app-root-state')

This block contains in JSON format the raw data used to generate various plots and tables on Weather Underground.  We need to extract this in a consistent way.  We will grab it as a string and clean it, then parse it as a JSON object (which is basically a `dict` containing `dict`s and `list`s).

In [ ]:
json_str = soup.find_all('script',id='app-root-state')[ 0 ].contents[ 0 ]
json_str = json_str.replace( '&q;','"' )

import json
import collections
weather_data = json.JSONDecoder(object_pairs_hook=collections.OrderedDict).decode( json_str )

In [ ]:
# Scan for set of observations (weather data).
for table in weather_data[ 'wu-next-state-key' ]:
    for observation in weather_data[ 'wu-next-state-key' ][ table ]:
        if 'observations' in weather_data[ 'wu-next-state-key' ][ table ][ observation ]:
            if type ( weather_data[ 'wu-next-state-key' ][ table ][ observation ] ) == str:
                continue
            for datum in weather_data[ 'wu-next-state-key' ][ table ][ observation ]:
                weather_data_block = weather_data[ 'wu-next-state-key' ][ table ][ observation ][ datum ]


In [ ]:
weather_data_block

In [ ]:
observations = {}
for observation in weather_data_block:
    observations[ observation[ 'obsTimeLocal' ] ] = observation[ 'imperial' ]
    observations[ observation[ 'obsTimeLocal' ] ][ 'winddirAvg' ] = int( observation[ 'winddirAvg' ] )

import pandas as pd
df = pd.DataFrame.from_dict( observations ).transpose()

In [ ]:
df.reset_index( level=0,inplace=True )
df = df.rename( columns={ 'index':'Date' } )
df

In [ ]:
df.columns

In [ ]:
import datetime

#date_offset = int( df[ 'Date' ][ 0 ] )
#date_origin = datetime.date.today() - datetime.timedelta( days=date_offset )

#df[ 'Date' ] = pd.to_numeric( df[ 'Date' ] )
df[ 'Date' ] = pd.to_datetime( df[ 'Date' ],format='%Y-%m-%d %H:%M:%S' )

In [ ]:
df[ 'tempAvg' ].plot( marker='.',linestyle='' )

Given this DataFrame, we can populate a Bokeh dashboard.  We can display many variables provided by the weather station, as shown above.  We'll start by concerning ourselves with only two:  we'll make a history plot of temperature and we'll make a directional vector for wind speed and direction.

In [ ]:
from bokeh.plotting import figure
from bokeh.models import HoverTool
from bokeh.io import push_notebook,output_notebook,show
output_notebook()

from ipywidgets import interact
import numpy as np

In [ ]:
x = df[ 'Date' ]
y = df[ 'tempAvg' ]

# This is a plot:
p_interact_temp = figure(
                    title="Temperature (°F)",
                    plot_height=300,plot_width=600,
                    y_range=(-20,+120),
                    x_axis_type="datetime"
                  )

# This is a line on that plot:
r_interact_temp = p_interact_temp.line( x,y,color="#ff5522",line_width=4 )

# These are settings for displaying information about the line:
HOVER = HoverTool(
    tooltips=[
        ("Time","@x{%m/%d %H:%M}"),
        ("Temperature", "@y°F")
    ],
    formatters={
        'x':'datetime'   # use 'datetime' formatter for 'Time' field
    },
    mode='vline'    # display a tooltip whenever the cursor is vertically in line with a glyph
)
p_interact_temp.add_tools( HOVER )

show( p_interact_temp,notebook_handle=True )

-   Copy the above plot to the cell below.  Modify it to show the total precipitation column.  (Change the labels and line color as well.)

In [ ]:
df

In [ ]:
df.columns

Next, let's make the windspeed vector display using a more refined technique than above.  Now we have actual bearings in degrees, so we can plot these data more precisely.

In [ ]:
x = df[ 'Date' ]
y1 = df[ 'winddirAvg' ]
y2 = df[ 'windspeedAvg' ]

# Convert the y-data into polar bearings.
import numpy as np
y1 = np.deg2rad( y1 )
xs = y2 * np.cos( y1 )
ys = y2 * np.sin( y1 )

# This is a plot:
p_interact_wind = figure(
                    title="Wind Speed and Direction",
                    plot_height=300,plot_width=300,
                    x_range=(-25,25),y_range=(-25,25)
                  )

# This is a line on that plot:
r_interact_wind = p_interact_wind.scatter( xs,ys,color="#22aaee" )
r_interact_wind_current = p_interact_wind.scatter( xs[ 0 ],ys[ 0 ],marker='x',color="#ff5500" )
r_interact_wind_origin = p_interact_wind.scatter( 0,0,marker='+',color="#000000" )

show( p_interact_wind,notebook_handle=True )

Given this much, we can compose these into a compound plot, or dashboard.

In [ ]:
from bokeh.layouts import gridplot

grid = gridplot( [ [ p_interact_temp,p_interact_wind ],[ None,None ] ] )
# subsequent rows would fill in the plots at the `None` locations above.

show( grid )

-   The following data are available and still unused in our dashboard:

    -   dew point
    -   atmospheric pressure
    -   precipitation
    -   solar radiation, uv index

    How would you choose to display each of these?  Implement two of them as new dashboard elements.